# 第 6 章｜RBAC + AES-256-GCM

依序執行每一格；可修改標示的參數後重跑。

In [ ]:
from pathlib import Path
import os, sys
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks": ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from course_utils import *
print("教材根目錄：", ROOT)
config = require_cillm_config()
print("執行模式：CILLM API（必要）")
print("GPT-OSS 模型：", config["model"])

> **執行模式 Hint**
>
> - 本教材每章都必須設定 `CILLM_API_KEY` 與 `CILLM_BASE_URL`，並呼叫 `openai/gpt-oss-120b`。
> - 缺少設定時會立即停止，不會以 mock 回覆取代真實模型。
> - 圖片解析另使用 `google/gemma-4-31b-it`，但沿用相同 CILLM API key。

> **角色 Hint**
>
> - `employee`：Excel Python Tool 與機務 Resource 應被拒絕。
> - `operations`：可分析 Excel 與讀航務規範，但不可讀機務規範。
> - `maintenance`：可讀機務 Resource；設定正確 AES key 後預期解密成功。
> - `developer`：可用動態 Python Tool 並讀資安規範。
> - `admin`：全部通過；改錯 AES key 仍應解密失敗。

RBAC 是以角色為基礎的存取控制；先授權，才執行 Tool 或解密 Resource。

In [ ]:
CURRENT_ROLE = "employee"  # 改成 operations / maintenance / developer / admin
print("目前角色：", CURRENT_ROLE)
show(ROLE_PERMISSIONS[CURRENT_ROLE])
route_plan = ask_gpt_oss("請分析大型航班 Excel。判斷可能需要什麼工具，但不要假裝已經執行。")
print("GPT-OSS 初步規劃：", route_plan)

## Tool 權限：執行前再次檢查。

In [ ]:
wanted_tool="excel_python_tool"
allowed=authorize(CURRENT_ROLE,"tools",wanted_tool)
if not allowed:
    result="權限不足：目前角色無法使用此工具"
else:
    result=safe_excel_python(ROOT/"data/excel/flight_delays.xlsx","df=pd.read_excel(excel_path)\nresult=len(df)")
print_execution_trace(question="請分析大型航班 Excel", role=CURRENT_ROLE, tool=wanted_tool, tool_auth="通過" if allowed else "拒絕", answer=result)

## AES-256-GCM Resource

GCM 同時加密並驗證是否遭竄改；每次使用隨機 nonce。先在 `.env` 放入 `AES_KEY`。

In [ ]:
# 產生教學金鑰（只顯示一次後請自行放入 .env，勿提交正式金鑰）
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
import base64
print("教學用 AES_KEY：", base64.urlsafe_b64encode(AESGCM.generate_key(bit_length=256)).decode())

In [ ]:
RESOURCE_NAME="maintenance_guidelines"
source=ROOT/"resources/maintenance/maintenance_guidelines.txt"
encrypted=ROOT/"generated/maintenance_guidelines.enc"
key=os.getenv("AES_KEY","")
resource_allowed=authorize(CURRENT_ROLE,"resources",RESOURCE_NAME)
aes_status="未執行"
if not resource_allowed:
    answer="權限不足：目前角色無法讀取此資源"
elif not key:
    answer="請先在 .env 設定 AES_KEY，再重新啟動 Kernel。"
else:
    try:
        encrypt_resource(source,encrypted,key); content=decrypt_resource(encrypted,key)
        aes_status="解密成功"; answer=content.splitlines()[1]
    except Exception:
        aes_status="解密失敗：金鑰錯誤或檔案內容已被變更"; answer=aes_status
print_execution_trace(question="讀取機務維修規範", role=CURRENT_ROLE, resource=RESOURCE_NAME, resource_auth="通過" if resource_allowed else "拒絕", aes=aes_status, answer=answer)

RBAC 決定誰能存取；AES-256 保護被複製的檔案。沒有權限時，程式不會先解密。

### 小練習

切換 `CURRENT_ROLE`，或改用錯誤 AES key，重新執行並比較追蹤。